# Treinamento de Modelos: AutoML, MLflow

Nesta aula, daremos o passo seguinte no ciclo de vida de um projeto de IA ("MREps"). Agora que governamos nossas características na Feature Store, aprenderemos a rastrear, acelerar e escalar o treinamento de modelos de "Machine Learning" utilizando três abordagens complementares no Databricks:

1. **MLflow com "Autolog":** Rastreamento automático de parâmetros, métricas e artefatos sem esforço de codificação.
2. **Databricks AutoML:** Geração automatizada de modelos concorrentes com transparência de código (caixa-branca).

---

## O "Dataset" Iris com MLflow "Autolog"

Antes de usarmos ferramentas de automação, precisamos entender como o **MLflow** intercepta nosso treinamento tradicional para garantir a reprodutibilidade. O Unity Catalog agora atua nativamente como o nosso **Model Registry** (Repositório de Modelos).

In [0]:
%%capture
%pip install databricks
%pip install databricks-automl --upgrade
dbutils.library.restartPython()

## Vamos recriar a tabela do dataset iris com a coluna `id_flor`

In [0]:
# Catalogo e Esquema do dataset Iris
catalogo_origem = "workspace"
esquema_origem = "default"
tabela_iris_raw = f"{catalogo_origem}.{esquema_origem}.iris_dataset"

# Lendo os dados brutos que já estão no ambiente
df_iris = spark.table(tabela_iris_raw)

# Garanta que a tabela possua a chave primária (id_flor) e as colunas ajustadas.

# Registra o DataFrame atual como uma View temporária no Spark
df_iris.createOrReplaceTempView("vw_iris_raw")

# Executa a query SQL para construir a chave primária
df_iris_sql = spark.sql("""
    SELECT 
        CONCAT('flor_', ROW_NUMBER() OVER (ORDER BY (SELECT NULL))) AS id_flor,
        sepal_length,
        sepal_width,
        petal_length,
        petal_width,
        species 
    FROM vw_iris_raw
""")

In [0]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from databricks import feature_engineering
from databricks.feature_engineering import FeatureLookup
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Inicializando o cliente da Feature Store e montando o Training Set
fe = feature_engineering.FeatureEngineeringClient()
df_target = df_iris_sql.select("id_flor", "species")
tabela_feature = 'workspace.default.iris_features_table'


In [0]:

features_lookup = [
    FeatureLookup(
        table_name=tabela_feature,
        feature_names=["sepal_length", "sepal_width", "petal_length", "petal_width"],
        lookup_key="id_flor"
    )
]

training_set = fe.create_training_set(
    df=df_target,
    feature_lookups=features_lookup,
    label="species",
    exclude_columns=["id_flor"]
)


In [0]:
# Convertendo para Pandas para o ecossistema Scikit-Learn / XGBoost
df_train = training_set.load_df().toPandas()
X = df_train.drop(columns=["species"])
y_raw = df_train["species"]

# Convertendo os alvos de texto para inteiros (0, 1, 2)
le = LabelEncoder()
y = le.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [0]:
# Autolog Global
mlflow.sklearn.autolog(log_models=False)

# Treinando vários modelos
modelos_para_testar = {
    "Logistic_Regression": LogisticRegression(),
    "Decision_Tree": DecisionTreeClassifier(),
    "Random_Forest": RandomForestClassifier(),
    "Gradient_Boosting": GradientBoostingClassifier()
}

for nome_modelo, instancia_modelo in modelos_para_testar.items():
    
    with mlflow.start_run(run_name=f"Model_{nome_modelo}") as run:
        print(f"Treinando e avaliando: {nome_modelo}...")
        
        # O Autolog intercepta o treinamento e salva os parâmetros de cada arquitetura
        instancia_modelo.fit(X_train, y_train)
        
        # Calculando e registrando a acurácia no conjunto de teste
        acuracia_teste = instancia_modelo.score(X_test, y_test)
        mlflow.log_metric("test_accuracy", acuracia_teste)
        print(f"{nome_modelo} - Acurácia de Teste: {acuracia_teste:.4f}")
        
        # Registrando o modelo envelopado com os metadados da Feature Store no Unity Catalog
        fe.log_model(
            model=instancia_modelo,
            artifact_path=f"model_{nome_modelo.lower()}",
            flavor=mlflow.sklearn,
            training_set=training_set,
            registered_model_name=f"{catalogo}.{esquema}.iris_model_prod"
        )
